# AdaptiveSLM Training on Colab/Kaggle

This notebook runs the complete training pipeline:
1. **Pre-training** - Custom MobileLLM-style architecture from scratch
2. **Knowledge Distillation** - From Qwen2.5-7B-Instruct
3. **PAKD Fine-tuning** - Profile-Aware Knowledge Distillation (novel contribution)
4. **GGUF Export** - For on-device inference

**Hardware**: GPU required (T4/V100/A100 on Colab; Kaggle P100/T4)
**Time**: ~6-12 hours depending on GPU

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets accelerate bitsandbytes wandb tqdm pyyaml psutil 2>&1 | tail -5

In [ ]:
# Clone the repo
import os
if not os.path.exists('Adaptive-SLM'):
    !git clone https://github.com/YOUR_USERNAME/Adaptive-SLM.git 2>&1 | tail -3
%cd Adaptive-SLM/training

In [ ]:
# Prepare training data
!python prepare_cloud_data.py --phase all --output-dir ./data --pretrain-samples 100000 --distill-samples 50000 --pakd-samples 20000

In [ ]:
# Phase 1: Pre-training (run this cell, takes ~2-4h on T4)
!python train.py --phase pretrain --data-dir ./data 2>&1 | tee pretrain.log

In [ ]:
# Phase 2: Knowledge Distillation (run after pretraining, ~1-2h)
!python train.py --phase distill --data-dir ./data 2>&1 | tee distill.log

In [ ]:
# Phase 3: PAKD Fine-tuning (novel contribution, ~1h)
!python train.py --phase pakd --data-dir ./data 2>&1 | tee pakd.log

In [ ]:
# Phase 4: Export to GGUF
!python train.py --phase export --output-model ../models/adaptive_slm.gguf 2>&1 | tee export.log

In [ ]:
# Quantize to Q4_K_M (requires llama.cpp)
!git clone --depth 1 https://github.com/ggerganov/llama.cpp 2>&1 | tail -3
!cd llama.cpp && make -j$(nproc) 2>&1 | tail -5
!./llama.cpp/build/bin/llama-quantize ../models/adaptive_slm.gguf ../models/adaptive_slm-q4_k_m.gguf Q4_K_M 2>&1 | tail -10

In [ ]:
# Quick test inference
import sys
sys.path.insert(0, '../core/build')
# Test would go here after building C++ core